In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm 
import scipy.stats as stats
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
from scipy.stats import skew 
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from matplotlib.ticker import MultipleLocator
from sklearn.preprocessing import PowerTransformer


In [ ]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'
#fpath = 'C:\\Users\\gianl\\Desktop\\Actigraphy Sara'

In [ ]:
df = pd.read_excel(fpath + '\\7.0_database_variables.xlsx', sheet_name = 'Sheet1')

In [ ]:
# Rename columns
df = df.rename(columns={'location(ita=0,uk=1,usa=2)': 'location', 'week(1=free days)': 'weekday_type'})

In [ ]:
# box plot to verify the outliers
fig, ax = plt.subplots(1, 6, figsize=(29, 5))
sns.boxplot(data=df['midsleep_h_UTC'], ax=ax[0])
sns.boxplot(data=df['sleep_start_decimal_UTC'], ax=ax[1])
sns.boxplot(data=df['sleep_end_decimal_UTC'], ax=ax[2])
sns.boxplot(data=df['phase(sleepoffset-sunrise)'], ax=ax[3])
sns.boxplot(data=df['sleep_duration(h)_UTC'], ax=ax[4])
sns.boxplot(data=df['waso(min)'], ax=ax[5])

plt.show()

In [ ]:
# remove outliers
# criteria: zscore of 3 means that the data point is 3 standard deviations away from the mean
df = df[(np.abs(stats.zscore(df['midsleep_h_UTC'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_start_decimal_UTC'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_end_decimal_UTC'])) < 3)]
df = df[(np.abs(stats.zscore(df['phase(sleepoffset-sunrise)'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_duration(h)_UTC'])) < 3)]

# box plot to verify the outliers in midpoint, sleep onset, sleep offset, phase, and sleep duration
fig, ax = plt.subplots(1, 6, figsize=(29, 5))
sns.boxplot(data=df['midsleep_h_UTC'], ax=ax[0])
sns.boxplot(data=df['sleep_start_decimal_UTC'], ax=ax[1])
sns.boxplot(data=df['sleep_end_decimal_UTC'], ax=ax[2])
sns.boxplot(data=df['phase(sleepoffset-sunrise)'], ax=ax[3])
sns.boxplot(data=df['sleep_duration(h)_UTC'], ax=ax[4])
sns.boxplot(data=df['waso(min)'], ax=ax[5])

plt.show()

In [ ]:
# define the start date
start_date = pd.to_datetime('2022-02-01')

In [ ]:
# function to count the week of the year from the start date 2022-02-01
def calculate_week_of_year(start_datetime):
    return (start_datetime - start_date).days // 7 + 5

# apply the function to calculate the week of the year
df['week_of_year'] = df['date'].apply(calculate_week_of_year)

In [ ]:
# adjust 'week of the year' to start from 0
df['week_of_year'] = df['week_of_year'] - 37

In [ ]:
# rename the location column as 0=ITA, 1=UK
df['location'] = df['location'].map({0: 'ITA', 1: 'UK'})

# rename the weekday column as 0=work days, 1=free days
df['weekday_type'] = df['weekday_type'].map({0: 'work days', 1: 'free days'})

In [ ]:
# calculate the sleep duration for work days and free days
df['sleep_duration(h)_UTC'] = df['sleep_duration(h)_UTC'].astype(float)
df['sleep_duration_work_days_UTC'] = df['sleep_duration(h)_UTC'] * (df['weekday_type'] == 'work days')
df['sleep_duration_free_days_UTC'] = df['sleep_duration(h)_UTC'] * (df['weekday_type'] == 'free days')

In [ ]:
# filtered the midpoints by type of day of the week
# new dataframe with only the midpoints of the work days/free days
df_workdays = df[df['weekday_type'] == 'work days']
df_freedays = df[df['weekday_type'] == 'free days']

In [ ]:
# create a new df for weekly jetlag analysis
data_jetlag_UTC = df 

In [ ]:
# calculate the mean midpoint for each location, week and weekday
weekly_means_jetlag_UTC = data_jetlag_UTC.groupby(['location', 'week_of_year', 'weekday_type'])['midsleep_h_UTC'].mean().unstack()

In [ ]:
# calculate the jet lag for week and weekday
weekly_means_jetlag_UTC['jet lag_UTC'] = weekly_means_jetlag_UTC['free days'] - weekly_means_jetlag_UTC['work days']

In [ ]:
# add a column with the location to the weekly_means_jetlag_UTC
weekly_means_jetlag_UTC['location'] = weekly_means_jetlag_UTC.index.get_level_values(0)

In [ ]:
# save the weekly_means_jetlag_UTC to an excel file
weekly_means_jetlag_UTC.to_excel(fpath + '\\weekly_means_jetlag_UTC.xlsx')

In [ ]:
# rename columns
df = df.rename(columns={'sleep_duration(h)_UTC': 'sleep_duration_UTC'})
df = df.rename(columns={'phase(sleepoffset-sunrise)': 'phase'})

In [ ]:
# Adding a 'season' column to the dataset based on the 'date' column
# Defining seasons based on months: 
# Winter (Dec-Feb), Spring (Mar-May), Summer (Jun-Aug), Autumn (Sep-Nov)
def assign_season(date):
    month = pd.to_datetime(date).month
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

In [ ]:
# Applying the function to create a season column
df_workdays['season'] = df_workdays['date'].apply(assign_season)
df_freedays['season'] = df_freedays['date'].apply(assign_season)
df['season'] = df['date'].apply(assign_season)

In [ ]:
# create a new variable 'photoperiod' based on the location
# if column 'location' = 1 take the value from 'photoperiod (h, UK)' 
# if column 'location' = 0 then photoperiod (h, ITA)'
df['photoperiod'] = np.where(df['location'] == 'UK', df['photoperiod (h, UK)'], df['photoperiod (h, ITA)'])

In [ ]:
# add a column with the photoperiod for the UK and ITA
df_workdays.loc[df_workdays['location'] == 'UK', 'photoperiod'] = df_workdays['photoperiod (h, UK)'] 
df_workdays.loc[df_workdays['location'] == 'ITA', 'photoperiod'] = df_workdays['photoperiod (h, ITA)']

In [ ]:
# add the photoperiod column to the df_freedays
df_freedays.loc[df_freedays['location'] == 'UK', 'photoperiod'] = df_freedays['photoperiod (h, UK)']
df_freedays.loc[df_freedays['location'] == 'ITA', 'photoperiod'] = df_freedays['photoperiod (h, ITA)']

In [ ]:
# descpriptive statistics
all_descriptive = df.describe()
all_descriptive = all_descriptive.transpose()

In [ ]:
# add the index as a column
all_descriptive['variable'] = all_descriptive.index 

#reset the index
all_descriptive = all_descriptive.reset_index(drop=True)

In [ ]:
all_descriptive.to_excel(fpath + '\\all_descriptive.xlsx')

In [ ]:
# descpriptive statistics for ITA
descriptive_ita = df[df['location'] == 'ITA'].describe()
descriptive_ita = descriptive_ita.transpose()

In [ ]:
descriptive_ita.to_excel(fpath + '\\descriptive_ita.xlsx')

In [ ]:
# descpriptive statistics for UK
descriptive_uk = df[df['location'] == 'UK'].describe()
descriptive_uk = descriptive_uk.transpose()

In [ ]:
descriptive_uk.to_excel(fpath + '\\descriptive_uk.xlsx')

In [ ]:
# % of time spent in each location
count_location = df['location'].value_counts(normalize=True) * 100

In [ ]:
count_location

In [ ]:
# distribution of the midpoint, sleep onset, sleep offset, and sleep duration
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.histplot(df['sleep_start_decimal_UTC'].dropna(), kde=True)
plt.title("Distribution of sleep onset")
plt.xlabel("Sleep onset(h, UTC)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 2)
sns.histplot(df['sleep_end_decimal_UTC'].dropna(), kde=True)
plt.title("Distribution of wake up time")
plt.xlabel("Wake up time(h, UTC)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 3)
sns.histplot(df['sleep_duration_UTC'].dropna(), kde=True)
plt.title("Distribution of sleep duration")
plt.xlabel("sleep duration(h)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# distribution of phase and midpoint of sleep
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.histplot(df['phase'].dropna(), kde=True)
plt.title("Distribution of phase(wake up time-sunrise)")
plt.xlabel("Phase(h)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 2)
sns.histplot(data=df, x='midsleep_h_UTC', hue='weekday_type', kde=True)
plt.title("Distribution of midsleep")
plt.xlabel("midpoint(h, UTC)")
plt.ylabel("Frequency")

plt.subplot(1, 3, 3)
sns.histplot(data=df, x='waso(min)', kde=True)
plt.title("Distribution of WASO")
plt.xlabel("WASO(min)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# test normality of the data using Shapiro-Wilk test 
# Null hipotesis(H0): data is normally distributed
shapiro_test_sleep_duration = stats.shapiro(df['sleep_duration_UTC'])
shapiro_test_midsleep = stats.shapiro(df['midsleep_h_UTC'])
shapiro_test_sleep_start = stats.shapiro(df['sleep_start_decimal_UTC'])
shapiro_test_sleep_end = stats.shapiro(df['sleep_end_decimal_UTC'])
shapiro_test_phase = stats.shapiro(df['phase'])
shapiro_test_waso = stats.shapiro(df['waso(min)'].dropna())

In [ ]:
shapiro_results_x = pd.DataFrame({
    'Variable': ['sleep_duration(h)', 'midsleep_h_UTC', 'sleep_start_decimal_UTC', 'sleep_end_decimal_UTC', 'phase(sleepoffset-sunrise)', 'waso(min)'],
    'Shapiro-Wilk test': [shapiro_test_sleep_duration, shapiro_test_midsleep, shapiro_test_sleep_start, shapiro_test_sleep_end, shapiro_test_phase, shapiro_test_waso]
})

In [ ]:
shapiro_results_x

In [ ]:
# test normality of the data using Shapiro-Wilk test for work days and free days (midpoint of sleep)
shapiro_test_midpoint_free = stats.shapiro(df[df['weekday_type'] == 'free days']['midsleep_h_UTC'])
shapiro_test_midpoint_work = stats.shapiro(df[df['weekday_type'] == 'work days']['midsleep_h_UTC'])

In [ ]:
shapiro_results_free_work = pd.DataFrame({
    'Weekday type': ['free days', 'work days'],
    'Shapiro-Wilk test': [shapiro_test_midpoint_free, shapiro_test_midpoint_work]
})

In [ ]:
shapiro_results_free_work

__General trends and variability in sleep patterns__

In [ ]:
plt.figure(figsize=(15, 15))
plt.subplot(4, 1, 1)
sns.lineplot(data=df, x='date', y='midsleep_h_UTC', hue='location')
plt.title("Midsleep")
plt.xlabel("Date")
plt.ylabel("Midsleep(h, UTC)")
plt.xticks(rotation=45)
plt.gca().xaxis.set_major_locator(MultipleLocator(7))

plt.subplot(4, 1, 2)
sns.lineplot(data=df, x='date', y='sleep_start_decimal_UTC', hue='location')
plt.title("Sleep onset")
plt.xlabel("Date")
plt.ylabel("Sleep onset(h, UTC)")
plt.xticks(rotation=45)
plt.gca().xaxis.set_major_locator(MultipleLocator(7))

plt.subplot(4, 1, 3)
sns.lineplot(data=df, x='date', y='sleep_end_decimal_UTC', hue='location')
plt.title("Sleep offset")
plt.xlabel("Date")
plt.ylabel("Sleep offset(h, UTC)")
plt.xticks(rotation=45)
plt.gca().xaxis.set_major_locator(MultipleLocator(7))

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
sns.lineplot(data=df, x='date', y='sleep_duration_UTC', hue='location')
plt.title("Sleep duration")
plt.xlabel("Date")
plt.ylabel("Sleep duration(h)")
plt.xticks(rotation=45)
plt.gca().xaxis.set_major_locator(MultipleLocator(7))

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
sns.lineplot(data=df, x='date', y='waso(min)', hue='location')
plt.title("Wake After Sleep Onset")
plt.xlabel("Date")
plt.ylabel("WASO(min)")
plt.xticks(rotation=45)
plt.gca().xaxis.set_major_locator(MultipleLocator(7))

plt.tight_layout()
plt.show()

In [ ]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
plt.figure(figsize=(15, 10))
df_pivot = df.pivot_table(index='date', columns='location', values='midsleep_h_UTC')
sns.heatmap(df_pivot, cmap="viridis", cbar_kws={'label': 'Midsleep (UTC)'})
plt.title('Heatmap del Midsleep_h_UTC tra Italia e UK nel Tempo')
plt.xlabel('Località')
plt.ylabel('Data')

# Format the y-axis as dd-mm-aaaa
plt.gca().yaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y'))

plt.show()

__Sleep-wake pattern between Uk and Italy__

In [ ]:
# descriptive statistics by location
df_grouped_location = df.groupby('location').describe()
df_grouped_location = df_grouped_location.transpose()

In [ ]:
# compare the variables between ITA and UK
ttest_midsleep_all_loc = stats.ttest_ind(df[df['location'] == 'ITA']['midsleep_h_UTC'], df[df['location'] == 'UK']['midsleep_h_UTC'])
ttest_midsleep_workdays_loc = stats.ttest_ind(df_workdays[df_workdays['location'] == 'ITA']['midsleep_h_UTC'], df_workdays[df_workdays['location'] == 'UK']['midsleep_h_UTC'])
ttest_midsleep_freedays_loc = stats.ttest_ind(df_freedays[df_freedays['location'] == 'ITA']['midsleep_h_UTC'], df_freedays[df_freedays['location'] == 'UK']['midsleep_h_UTC'])
utest_duration_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['sleep_duration_UTC'], df[df['location'] == 'UK']['sleep_duration_UTC'])
utest_phase_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['phase'], df[df['location'] == 'UK']['phase'])
utest_start_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['sleep_start_decimal_UTC'], df[df['location'] == 'UK']['sleep_start_decimal_UTC'])
utest_end_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['sleep_end_decimal_UTC'], df[df['location'] == 'UK']['sleep_end_decimal_UTC'])
utest_waso_loc = stats.mannwhitneyu(df[df['location'] == 'ITA']['waso(min)'].dropna(), df[df['location'] == 'UK']['waso(min)'].dropna())

In [ ]:
# print the results
print('T test results by location')
print('Midsleep_all:', ttest_midsleep_all_loc)
print('Midsleep_work:', ttest_midsleep_workdays_loc)
print('Midsleep_free:', ttest_midsleep_freedays_loc)
print('U test results by location')
print('Sleep_onset:', utest_start_loc)
print('Sleep_offset:', utest_end_loc)
print('Sleep_duration:', utest_duration_loc)
print('Phase:', utest_phase_loc)
print('WASO:', utest_waso_loc)

In [ ]:
# plot the midpoint of sleep by location
plt.figure(figsize=(6, 5))
sns.boxplot(x='location', y='midsleep_h_UTC', data=df_workdays)
plt.title('Midsleep by location')
plt.suptitle('')  
plt.xlabel('')
plt.ylabel('Midsleep (h, UTC)')

plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)

plt.gca().yaxis.set_major_locator(MultipleLocator(0.5)) # gcd stands for 'get current axis'
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.tight_layout()

plt.show()

In [ ]:
# plot the midpoint of sleep by location for free days and work days
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

sns.boxplot(x='location', y='midsleep_h_UTC', data=df_workdays, ax=ax[0])
ax[0].set_title('Midsleep (work days) by location')
ax[0].set_ylabel('Midsleep (h, UTC)')
ax[0].set_xlabel('')
ax[0].yaxis.set_major_locator(MultipleLocator(0.5))
ax[0].set_ylim(22, 31)
ax[0].annotate('***', xy=(0.5, 0.9), xycoords='axes fraction', ha='center', va='center', fontsize=12) #add a significance line of ** for the p-value < 0.001

sns.boxplot(x='location', y='midsleep_h_UTC', data=df_freedays, ax=ax[1])
ax[1].set_title('Midsleep (free days) by location')
ax[1].set_ylabel('Midsleep (h, UTC)')
ax[1].set_xlabel('')
ax[1].yaxis.set_major_locator(MultipleLocator(0.5))
ax[1].set_ylim(ax[0].get_ylim())
ax[1].annotate('***', xy=(0.5, 0.9), xycoords='axes fraction', ha='center', va='center', fontsize=12) #add a significance line of  for the p-value < 0.001
    
sns.despine()
plt.grid(False)
plt.gca().spines['bottom'].set_color('black') # set the color of the x-axis to black
plt.gca().spines['left'].set_color('black')
plt.tight_layout()
plt.show()

In [ ]:
# plot the sleep onset and sleep offset by location
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x='location', y='sleep_start_decimal_UTC', data=df, gap=0.2)
plt.title('Mean sleep onset by location')
plt.xlabel('')
plt.ylabel('sleep onset (h, UTC)')
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12) #add a significance line to the subplot
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

plt.subplot(1, 2, 2)
sns.boxplot(x='location', y='sleep_end_decimal_UTC', data=df, gap=0.2)
plt.title('Mean wake up time by location')
plt.xlabel('')
plt.ylabel('sleep offset (h, UTC)')
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12) #add a significance line to the subplot
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# plot the phase (sleep offset-sunrise) by location 
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x='location', y='phase', data=df, gap=0.2)
plt.title('Mean of phase (wake up time-sunrise) by location')
plt.xlabel('')
plt.ylabel('phase (h)')
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# plot the waso by location
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x='location', y='waso(min)', data=df, gap=0.2)
plt.title('Mean of Wake After Sleep Onset by location')
plt.xlabel('')
plt.ylabel('WASO (min)')
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
#plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

sns.despine()
plt.tight_layout()
plt.show()

__Jet lag__

In [ ]:
# drop the rows with missing values
jetlag_clean = weekly_means_jetlag_UTC['jet lag_UTC'].dropna()

In [ ]:
# Test normality of the jet lag data using Shapiro-Wilk test 
# H0: data is normally distributed
shapiro_test_jetlag = stats.shapiro(jetlag_clean)

In [ ]:
print('Shapiro test result for jet lag:')
print(shapiro_test_jetlag)

In [ ]:
# test the difference in jet lag between the two locations
utest_jetlag = stats.mannwhitneyu(weekly_means_jetlag_UTC[weekly_means_jetlag_UTC['location'] == 'ITA']['jet lag_UTC'].dropna(), weekly_means_jetlag_UTC[weekly_means_jetlag_UTC['location'] == 'UK']['jet lag_UTC'].dropna())

In [ ]:
print('U test result for jet lag by location:')
print(utest_jetlag)

__Season and sleep-wake pattern__

In [ ]:
# remove NaN values from the columns and create a new dataframe
df1 = df.dropna(subset=['sleep_duration_UTC']) 
df1 = df.dropna(subset=['phase'])

In [ ]:
ols_midsleep_season = ols('midsleep_h_UTC ~ C(season)', data=df1).fit() # generate and fit the regression model
anova_midsleep_result_season = sm.stats.anova_lm(ols_midsleep_season, typ=3) # fit the ANOVA model and get the results

In [ ]:
print('ANOVA Result:')
print(anova_midsleep_result_season)

In [ ]:
# Post hoc test: perform a Tukey HSD test to compare the means
tukey_results_season1 = pairwise_tukeyhsd(df1['midsleep_h_UTC'], df1['season'])
print(tukey_results_season1)

In [ ]:
# sleep midpoint by season
plt.figure(figsize=(12, 6))
sns.boxplot(x='season', y='midsleep_h_UTC', data=df1)
plt.title('Midsleep by season')
plt.suptitle('')  # add space between the title and the plot
plt.ylabel('Midsleep (h, UTC)')
plt.xlabel('')
sns.despine()
plt.grid(False)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.gca().spines['bottom'].set_color('black') 
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
# Drop rows with missing values in 'waso(min)' column
df_waso = df1.dropna(subset=['waso(min)'])

In [ ]:
# Kruskal-Wallis test : compare the sleep duration, phase, sleep onset, sleep offset and waso(min) between the seasons
kw_sleep_duration_season = stats.kruskal(df[df['season'] == 'Winter']['sleep_duration_UTC'], df[df['season'] == 'Spring']['sleep_duration_UTC'], df[df['season'] == 'Summer']['sleep_duration_UTC'], df[df['season'] == 'Autumn']['sleep_duration_UTC'])
kw_phase_season = stats.kruskal(df[df['season'] == 'Winter']['phase'], df[df['season'] == 'Spring']['phase'], df[df['season'] == 'Summer']['phase'], df[df['season'] == 'Autumn']['phase'])
kw_start_season = stats.kruskal(df[df['season'] == 'Winter']['sleep_start_decimal_UTC'], df[df['season'] == 'Spring']['sleep_start_decimal_UTC'], df[df['season'] == 'Summer']['sleep_start_decimal_UTC'], df[df['season'] == 'Autumn']['sleep_start_decimal_UTC'])
kw_end_season = stats.kruskal(df[df['season'] == 'Winter']['sleep_end_decimal_UTC'], df[df['season'] == 'Spring']['sleep_end_decimal_UTC'], df[df['season'] == 'Summer']['sleep_end_decimal_UTC'], df[df['season'] == 'Autumn']['sleep_end_decimal_UTC'])
kw_end_season = stats.kruskal(df_waso[df_waso['season'] == 'Winter']['waso(min)'], df_waso[df_waso['season'] == 'Spring']['waso(min)'], df_waso[df_waso['season'] == 'Summer']['waso(min)'], df_waso[df_waso['season'] == 'Autumn']['waso(min)'])

In [ ]:
print('Results for Kruskal-Wallis test for the sleep duration by season')
print(kw_sleep_duration_season)
print('Results for Kruskal-Wallis test for the phase by season')
print(kw_phase_season)
print('Results for Kruskal-Wallis test for the sleep onset by season')
print(kw_start_season)
print('Results for Kruskal-Wallis test for the sleep offset by season')
print(kw_end_season)
print('Results for Kruskal-Wallis test for the waso by season')
print(kw_end_season)

In [ ]:
# phase (sleep offset - sunrise) by season
plt.figure(figsize=(12, 6))
sns.boxplot(x='season', y='phase', data=df)
plt.title('Phase (wake up time-sunrise) by season')
plt.suptitle('')  # add space after the title
plt.ylabel('Phase (h)')
plt.xlabel('')
sns.despine()
plt.grid(False)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
# Post hoc test: perform a Tukey HSD test to compare the means
tukey_results_season2 = pairwise_tukeyhsd(df1['phase'], df1['season'])
print(tukey_results_season2)

In [ ]:
# sleep onset and offset by season
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.boxplot(x='season', y='sleep_start_decimal_UTC', data=df)
plt.title('Sleep onset by season')
plt.ylabel('Sleep onset (h, UTC)')
plt.xlabel('')
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

plt.subplot(1, 2, 2)
sns.boxplot(x='season', y='sleep_end_decimal_UTC', data=df)
plt.title('Wake up time by season')
plt.ylabel('Sleep offset (h, UTC)')
plt.xlabel('')
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.tight_layout()

plt.show()

In [ ]:
# Post hoc test: perform a Tukey HSD test to compare the means
tukey_results_season3 = pairwise_tukeyhsd(df1['sleep_start_decimal_UTC'], df1['season'])
print(tukey_results_season3)

In [ ]:
# Post hoc test: perform a Tukey HSD test to compare the means
tukey_results_season4 = pairwise_tukeyhsd(df1['sleep_end_decimal_UTC'], df1['season'])
print(tukey_results_season4)

In [ ]:
# waso by season
plt.figure(figsize=(12, 6))
sns.boxplot(x='season', y='waso(min)', data=df_waso)
plt.title('WASO by season')
plt.ylabel('WASO (min)')
plt.xlabel('')
plt.gca().yaxis.set_major_locator(MultipleLocator(30))

plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)

plt.show()

In [ ]:
# Post hoc test: perform a Tukey HSD test to compare the means
tukey_results_season5 = pairwise_tukeyhsd(df_waso['waso(min)'], df_waso['season'])
print(tukey_results_season5)

__DST and sleep-wake pattern__

In [ ]:
# rename the column
df = df.rename(columns={'DST(0=ST)': 'DST_0'})

In [ ]:
# remove the rows with missing values in the columns
df2 = df.dropna(subset=['sleep_duration_UTC'])
df2 = df.dropna(subset=['phase'])

In [ ]:
# t-test to compare the midpoint of sleep between DST and non-DST
ttest_midsleep_dst = stats.ttest_ind(df2[df2['DST_0'] == 0]['midsleep_h_UTC'], df2[df2['DST_0'] == 1]['midsleep_h_UTC'])

In [ ]:
print('T test result for the sleep midpoint by DST:')
print(ttest_midsleep_dst)

In [ ]:
# sleep midpoint by DST
plt.figure(figsize=(8, 5))
sns.boxplot(x='DST_0', y='midsleep_h_UTC', data=df2)
plt.title('Midsleep by DST')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Midsleep (h, UTC)')
plt.xlabel('')
plt.xticks([0, 1], ['DST', 'ST'])
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
sns.despine()
plt.grid(False)

plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
# Mann Whitney U test : compare the sleep duration, phase, sleep onset, and sleep offset between DST and non-DST
utest_sleep_duration_dst = stats.mannwhitneyu(df2[df2['DST_0'] == 0]['sleep_duration_UTC'], df2[df2['DST_0'] == 1]['sleep_duration_UTC'])
utest_phase_dst = stats.mannwhitneyu(df2[df2['DST_0'] == 0]['phase'], df2[df2['DST_0'] == 1]['phase'])
utest_start_dst = stats.mannwhitneyu(df2[df2['DST_0'] == 0]['sleep_start_decimal_UTC'], df2[df2['DST_0'] == 1]['sleep_start_decimal_UTC'])
utest_end_dst = stats.mannwhitneyu(df2[df2['DST_0'] == 0]['sleep_end_decimal_UTC'], df2[df2['DST_0'] == 1]['sleep_end_decimal_UTC'])
utest_waso_dst = stats.mannwhitneyu(df2[df2['DST_0'] == 0]['waso(min)'].dropna(), df2[df2['DST_0'] == 1]['waso(min)'].dropna())

In [ ]:
print('Results for Mann-Whitney U test for the sleep duration by dst')
print(utest_sleep_duration_dst)
print('Results for Mann-Whitney U test for the phase by dst')
print(utest_phase_dst)
print('Results for Mann-Whitney U test for the sleep onset by dst')
print(utest_start_dst)
print('Results for Mann-Whitney U test for the sleep offset by dst')
print(utest_end_dst)
print('Results for Mann-Whitney U test for the waso by dst')
print(utest_waso_dst)

In [ ]:
# phase (sleep offset - sunrise) by DST
plt.figure(figsize=(6, 4))
sns.boxplot(x='DST_0', y='phase', data=df2)
plt.title('Phase (wake up time-sunrise) by DST')
plt.suptitle('')  # Removing default subtitle
plt.ylabel('Phase (h)')
plt.xlabel('')
plt.xticks([0, 1], ['DST', 'ST'])
sns.despine()
plt.grid(False)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.annotate('***', xy=(0.49, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.show()

In [ ]:
# sleep onset and offset by DST
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.boxplot(x='DST_0', y='sleep_start_decimal_UTC', data=df2)
plt.title('Sleep onset by DST')
plt.ylabel('Sleep onset (h, UTC)')
plt.xlabel('')
plt.xticks([0, 1], ['DST', 'ST'])
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

plt.subplot(1, 2, 2)
sns.boxplot(x='DST_0', y='sleep_end_decimal_UTC', data=df2)
plt.title('Wake up time by DST')
plt.ylabel('Wake up time (h, UTC)')
plt.xlabel('')
plt.xticks([0, 1], ['DST', 'ST'])
plt.annotate('***', xy=(0.5, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)
plt.tight_layout()

plt.show()

__Photoperiod and sleep-wake pattern__

In [ ]:
# correlation between sleep-wake variables and photoperiod
correlation_test1 = stats.pearsonr(df['midsleep_h_UTC'], df['photoperiod'])
correlation_test2 = stats.pearsonr(df_workdays['midsleep_h_UTC'], df_workdays['photoperiod'])
correlation_test3 = stats.pearsonr(df_freedays['midsleep_h_UTC'], df_freedays['photoperiod'])
correlation_test4 = stats.spearmanr(df['sleep_duration_UTC'], df['photoperiod'])
correlation_test5 = stats.spearmanr(df['sleep_end_decimal_UTC'], df['photoperiod'])
correlation_test6 = stats.spearmanr(df_waso['waso(min)'], df_waso['photoperiod'])

In [ ]:
# extract the coefficients and p-values from the correlation test results
correlation_coeff = [correlation_test1.statistic, correlation_test2.statistic, correlation_test3.statistic, correlation_test4.statistic, correlation_test5.statistic, correlation_test6.statistic]
p_values = [correlation_test1.pvalue, correlation_test2.pvalue, correlation_test3.pvalue, correlation_test4.pvalue, correlation_test5.pvalue, correlation_test6.pvalue]

In [ ]:
# create a DataFrame with the results
correlation_results = pd.DataFrame({
    'Variables': ['midsleep UTC vs photoperiod', 'midsleep (work) UTC vs photoperiod', 'midsleep (free) UTC vs photoperiod', 'sleep duration UTC vs photoperiod', 'sleep offset UTC vs photoperiod', 'WASO(min) vs photoperiod'],
    'Coefficient': correlation_coeff,
    'P-value': p_values
})

correlation_results

In [ ]:
# plot the correlation between the midpoint of sleep and photoperiod, for all the data, work days and free days
plt.figure(figsize=(16, 6))

# All data
plt.subplot(1, 4, 1)
sns.scatterplot(x='midsleep_h_UTC', y='photoperiod', data=df)
sns.regplot(x='midsleep_h_UTC', y='photoperiod', data=df, scatter=False, color='red')
corr_all, _ = stats.pearsonr(df['midsleep_h_UTC'], df['photoperiod'])
plt.title(f'Midsleep vs photoperiod\nCorrelation: {corr_all:.2f}')
plt.xlabel('Midsleep (h, UTC)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)

# Work days
plt.subplot(1, 4, 2)
sns.scatterplot(x='midsleep_h_UTC', y='photoperiod', data=df_workdays)
sns.regplot(x='midsleep_h_UTC', y='photoperiod', data=df_workdays, scatter=False, color='red')
corr_work, _ = stats.pearsonr(df_workdays['midsleep_h_UTC'], df_workdays['photoperiod'])
plt.title(f'Midsleep (work) vs photoperiod\nCorrelation: {corr_work:.2f}')
plt.xlabel('Midsleep (h, UTC)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)

# Free days
plt.subplot(1, 4, 3)
sns.scatterplot(x='midsleep_h_UTC', y='photoperiod', data=df_freedays)
sns.regplot(x='midsleep_h_UTC', y='photoperiod', data=df_freedays, scatter=False, color='red')
corr_free, _ = stats.pearsonr(df_freedays['midsleep_h_UTC'], df_freedays['photoperiod'])
plt.title(f'Midsleep (free) vs photoperiod\nCorrelation: {corr_free:.2f}')
plt.xlabel('Midsleep (h, UTC)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)

# Sleep offset
plt.subplot(1, 4, 4)
sns.scatterplot(x='sleep_end_decimal_UTC', y='photoperiod', data=df)
sns.regplot(x='sleep_end_decimal_UTC', y='photoperiod', data=df, scatter=False, color='red')
corr_offset, _ = stats.pearsonr(df['sleep_end_decimal_UTC'], df['photoperiod'])
plt.title(f'Sleep offset vs photoperiod\nCorrelation: {corr_offset:.2f}')
plt.xlabel('Sleep offset (h, UTC)')
plt.ylabel('Photoperiod (h)')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
sns.despine()
plt.grid(False)

plt.tight_layout()
plt.show()

__Weekly IV, IS and RA__

In [ ]:
# load the datasets required for the analysis
weekly_values = pd.read_csv(fpath + '\\2.0_weekly_IV_IS_RA_values_with_dates.csv')
weekly_jetlag = weekly_means_jetlag_UTC

In [ ]:
# split Date_Range into Start_Date and End_Date in weekly_values
weekly_values[['Start_Date', 'End_Date']] = weekly_values['Date_Range'].str.split(' to ', expand=True)

In [ ]:
# convert Start_Date and End_Date to datetime format
weekly_values['Start_Date'] = pd.to_datetime(weekly_values['Start_Date'])
weekly_values['End_Date'] = pd.to_datetime(weekly_values['End_Date'])

In [ ]:
# merge by matching the week number extracted from Start_Date with week_of_year in weekly_jetlag
merged_data = pd.merge(
    weekly_values,
    weekly_jetlag,
    left_on=weekly_values['Start_Date'].dt.isocalendar().week,
    right_on='week_of_year',
    how='inner'
)

In [ ]:
merged_data.head()

In [ ]:
# box plot to verify the outliers in IV, IS, and RA
fig, ax = plt.subplots(1, 3, figsize=(24, 7))
sns.boxplot(data=merged_data['IV'], ax=ax[0])
sns.boxplot(data=merged_data['IS'], ax=ax[1])
sns.boxplot(data=merged_data['RA'], ax=ax[2])
plt.show()

In [ ]:
# summary statistics
summary_stats = merged_data.groupby("location")[['IV', 'IS', 'RA']].describe()
summary_stats

In [ ]:
# distribution of IV, IS, and RA
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
sns.histplot(merged_data['IV'].dropna(), kde=True)
plt.title('Distribution of IV')
plt.xlabel('IV')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
sns.histplot(merged_data['IS'].dropna(), kde=True)
plt.title('Distribution of IS')
plt.xlabel('IS')
plt.ylabel('Frequency')
 
plt.subplot(1, 3, 3)
sns.histplot(merged_data['RA'].dropna(), kde=True)
plt.title('Distribution of RA')
plt.xlabel('RA')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# compare the variables between ITA and UK
iv_ttest = stats.ttest_ind(merged_data[merged_data['location'] == 'ITA']['IV'], merged_data[merged_data['location'] == 'UK']['IV'], nan_policy='omit')
is_utest = stats.mannwhitneyu(merged_data[merged_data['location'] == 'ITA']['IS'], merged_data[merged_data['location'] == 'UK']['IS'], nan_policy='omit')
ra_utest = stats.mannwhitneyu(merged_data[merged_data['location'] == 'ITA']['RA'], merged_data[merged_data['location'] == 'UK']['RA'], nan_policy='omit')

In [ ]:
print('Test results for IV by location:', iv_ttest)
print('Test results for IS by location:', is_utest)
print('Test results for RA by location:', ra_utest)

------------------------

__Sleep-wake patterns over time__

_Midsleep_

In [ ]:
# n of days by day_after_flight
count_day_after_flight = df['day_after_flight'].value_counts()

In [ ]:
# filter the data by the days after the flight (8)
sleep_bydays_after_flight = df[df['day_after_flight'] <= 15]

In [ ]:
# download the data df to a csv file
df.to_csv(fpath + '\\df1.csv', index=False)

# download the sleep_bydays_after_flight data to a csv file
sleep_bydays_after_flight.to_csv(fpath + '\\sleep_bydays_after_flight1.csv', index=False)

In [ ]:
# group by location
sleep_bydays_after_flight_uk = sleep_bydays_after_flight[sleep_bydays_after_flight['location'] == 'UK']
sleep_bydays_after_flight_ita = sleep_bydays_after_flight[sleep_bydays_after_flight['location'] == 'ITA']

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(sleep_bydays_after_flight_uk['day_after_flight'], sleep_bydays_after_flight_uk['midsleep_h_UTC'], label='UK', color='blue')
plt.scatter(sleep_bydays_after_flight_ita['day_after_flight'], sleep_bydays_after_flight_ita['midsleep_h_UTC'], label='ITA', color='red')
plt.xlabel('Day After Flight')
plt.ylabel('Midsleep (h, UTC)')
plt.gca().xaxis.set_major_locator(MultipleLocator(1))
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.title('Midsleep by Day After Flight')
plt.legend()
plt.show()

In [ ]:
# new column with the day after flight category
df['day_after_flight_category'] = pd.cut(df['day_after_flight'], bins=[0, 5, 10, 15, max(df['day_after_flight'])], labels=['1-3 days', '4-6 days','7-9 days', '>10 days'])

In [ ]:
# new column with the day after flight category 1-3 days and >3 days
df['day_after_flight_category_simple'] = pd.cut(df['day_after_flight'], bins=[0, 3, max(df['day_after_flight'])], labels=['1-3 days', '>3 days'])

In [ ]:
# linear regression model with categorical variables
model1a = smf.ols('midsleep_h_UTC ~ C(day_after_flight_category)', data=df).fit()

print(model1a.summary())

In [ ]:
# linear regression model with categorical variables
model1b = smf.ols('midsleep_h_UTC ~ C(day_after_flight_category_simple)', data=df, re_formula='~1').fit()

print(model1b.summary())

In [ ]:
model1c = smf.mixedlm('midsleep_h_UTC ~ location * day_after_flight', df, groups=df["day_after_flight"], re_formula="~1").fit()

print(model1c.summary())

In [ ]:
# Fit a mixed-effects model with random intercepts for each day after flight
model2a = smf.mixedlm('midsleep_h_UTC ~ C(location) + photoperiod + C(DST_0)', data=df, groups=df['day_after_flight'], re_formula='~1').fit()

print(model2a.summary())

In [ ]:
# Fit a mixed-effects model with random intercepts for each date and day after flight
model2b = smf.mixedlm('midsleep_h_UTC ~ C(location) + photoperiod + C(DST_0) + C(location)*photoperiod', data=df, groups=df['day_after_flight'], re_formula='~1').fit()

print(model2b.summary())

In [ ]:
# Fit a mixed-effects model with random intercepts for each date and day after flight
model2c = smf.mixedlm('midsleep_h_UTC ~ C(day_after_flight_category) + C(location) + photoperiod + C(DST_0) + C(location)*photoperiod', data=df, groups=df['day_after_flight'], re_formula='~1').fit()

print(model2c.summary())

In [ ]:
# Ordiniamo i dati per data per rispettare la struttura temporale
df = df.sort_values(by="date")

# Definiamo la struttura di correlazione per GEE (AR1 = autoregressivo di primo ordine)
independent_var = sm.cov_struct.Autoregressive()

# Modello GEE
gee_model = smf.gee(
    "midsleep_h_UTC ~ location + DST_0 + photoperiod + day_after_flight_category",
    data=df,
    groups=df["date"],  # Usa la data come gruppo per catturare la dipendenza temporale
    cov_struct=independent_var,
    family=sm.families.Gaussian()
).fit()

# Mostriamo il riassunto del modello GEE
gee_model.summary()


In [ ]:
coef_df2 = pd.DataFrame({'coef': model2b.params.values, 'p-value': model2b.pvalues.values, '0.025': model2b.conf_int()[0], '0.975': model2b.conf_int()[1]})
coef_df2

In [ ]:
#drop non significant variables
coef_df2 = coef_df2.drop('Intercept')
coef_df2 = coef_df2.drop('Group Var', axis=0)

In [ ]:
# initialize the matplotlib figure
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

ax = sns.stripplot(x="coef", y=coef_df2.index, data=coef_df2, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)
# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_df2.shape[0]):
    plt.plot([coef_df2['0.025'].iloc[i], coef_df2['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_df2['0.025'].iloc[i], coef_df2['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_df2['0.975'].iloc[i], coef_df2['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add the significance as *** for p-value < 0.001, ** for p-value < 0.01, * for p-value < 0.05 close to the variable name
for i in range(coef_df2.shape[0]):
    if coef_df2['p-value'].iloc[i] < 0.001:
        plt.text(coef_df2['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_df2['p-value'].iloc[i] < 0.01:
        plt.text(coef_df2['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_df2['p-value'].iloc[i] < 0.05:
        plt.text(coef_df2['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9) 

# add a legend of the significance at the top of the plot
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('LMM (outcome=midsleep)', fontsize=20, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.5)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


ax.set_yticklabels(['Location[T.UK]', 'Time shift[T.ST]', 'Photoperiod', 'Location[T.UK]*Photoperiod', ])

plt.show()

In [ ]:
# Shapiro-Wilk test for the residuals   
shapiro_residuals2 = stats.shapiro(model2b.resid)

print('Shapiro test for the residuals:', shapiro_residuals2)

In [ ]:
residuals2 = model2b.resid 
predicted2 = model2b.fittedvalues

In [ ]:
# Breusch-Pagan test for homoscedasticity
bp_test2 = het_breuschpagan(residuals2, model2b.model.exog)

# results of the Breusch-Pagan test
bp_stat2, bp_pval2, _, _ = bp_test2
print(f'Breusch-Pagan statistic: {bp_stat2}, p-value: {bp_pval2}')
if bp_pval2 > 0.05:
    print('The residuals are homoscedastic (fail to reject H0).')
else:
    print('The residuals are heteroscedastic (reject H0).')

In [ ]:
# Durbin-Watson test for autocorrelation
durbin_watson_test2 = durbin_watson(residuals2)

print('Durbin-Watson test:', durbin_watson_test2)

In [ ]:
# The Durbin-Watson test statistic is close to 2, which indicates that there is no significant autocorrelation in the residuals

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='midsleep_h_UTC', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'])
plt.title('Midsleep by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Midsleep (h, UTC)', fontsize=12, fontweight='bold')
plt.xlabel('')
plt.legend()
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.legend(title='Location', labels=['ITA', 'UK'], loc='upper right', handles=[plt.Line2D([0], [0], color='darkblue', lw=2), plt.Line2D([0], [0], color='darkorange')])
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST 
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

# mean of the phase for the two locations
#plt.axhline(y=df[df['location'] == 'ITA']['phase'].mean(), color='darkblue', linestyle='--', label='Mean ITA')
#plt.text(plt.xlim()[1], df[df['location'] == 'ITA']['phase'].mean(), 'Mean ITA', ha='right', va='bottom')
#plt.axhline(y=df[df['location'] == 'UK']['phase'].mean(), color='darkorange', linestyle='--', label='Mean UK')
#plt.text(plt.xlim()[1], df[df['location'] == 'UK']['phase'].mean(), 'Mean UK', ha='right', va='bottom')

plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='midsleep_h_UTC', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'])
plt.title('Midsleep by location over time (2023)', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Midsleep (h, UTC)', fontsize=12, fontweight='bold')
plt.xlabel('')
plt.legend()
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min() + pd.DateOffset(months=3), df['date'].min() + pd.DateOffset(months=15))
plt.legend(title='Location', labels=['ITA', 'UK'], loc='upper right', handles=[plt.Line2D([0], [0], color='darkblue', lw=2), plt.Line2D([0], [0], color='darkorange')])
plt.gca().xaxis.set_major_locator(MultipleLocator(8)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST 
#plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
#plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
#plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

# mean of the phase for the two locations
#plt.axhline(y=df[df['location'] == 'ITA']['phase'].mean(), color='darkblue', linestyle='--', label='Mean ITA')
#plt.text(plt.xlim()[1], df[df['location'] == 'ITA']['phase'].mean(), 'Mean ITA', ha='right', va='bottom')
#plt.axhline(y=df[df['location'] == 'UK']['phase'].mean(), color='darkorange', linestyle='--', label='Mean UK')
#plt.text(plt.xlim()[1], df[df['location'] == 'UK']['phase'].mean(), 'Mean UK', ha='right', va='bottom')

plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='midsleep_h_UTC', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'])
plt.title('Midsleep by location over time (2024)', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Midsleep (h, UTC)', fontsize=12, fontweight='bold')
plt.xlabel('')
plt.legend()
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min() + pd.DateOffset(months=15), df['date'].min() + pd.DateOffset(months=27))
plt.legend(title='Location', labels=['ITA', 'UK'], loc='upper right', handles=[plt.Line2D([0], [0], color='darkblue', lw=2), plt.Line2D([0], [0], color='darkorange')])
plt.gca().xaxis.set_major_locator(MultipleLocator(8)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST 
#plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
#plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
#plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

# mean of the phase for the two locations
#plt.axhline(y=df[df['location'] == 'ITA']['phase'].mean(), color='darkblue', linestyle='--', label='Mean ITA')
#plt.text(plt.xlim()[1], df[df['location'] == 'ITA']['phase'].mean(), 'Mean ITA', ha='right', va='bottom')
#plt.axhline(y=df[df['location'] == 'UK']['phase'].mean(), color='darkorange', linestyle='--', label='Mean UK')
#plt.text(plt.xlim()[1], df[df['location'] == 'UK']['phase'].mean(), 'Mean UK', ha='right', va='bottom')

plt.show()

Wake After Sleep Onset

In [ ]:
df.rename(columns={'waso(min)': 'waso_min'}, inplace=True)
sleep_bydays_after_flight.rename(columns={'waso(min)': 'waso_min'}, inplace=True)

In [ ]:
# drop the nan values in the waso_min column
df_waso_clean = df.dropna(subset=['waso_min'])

In [ ]:
#plot waso vs day after flight
plt.figure(figsize=(10, 6))
sns.scatterplot(x='day_after_flight', y='waso_min', hue='location', data=df, palette=['darkblue', 'darkorange'])
plt.title('WASO by days after flight', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('WASO (min)', fontsize=12, fontweight='bold')
plt.xlabel('Days After Flight')
plt.grid(True)
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
#plt.gca().xaxis.set_major_locator(MultipleLocator(1))
plt.gca().yaxis.set_major_locator(MultipleLocator(15))

plt.show()

In [ ]:
# Calcoliamo lo Z-score per la variabile 'waso'
df_waso_clean['z_waso'] = np.abs(stats.zscore(df_waso_clean['waso_min']))

# Filtriamo il dataset per rimuovere gli outlier (Z-score > 3 o < -3)
df_no_outliers = df_waso_clean[(df_waso_clean['z_waso'] <= 3) & (df_waso_clean['z_waso'] >= -3)].drop(columns=['z_waso'])

# Verifichiamo il numero di righe prima e dopo la rimozione degli outlier
original_count = df_waso_clean.shape[0]
filtered_count = df_no_outliers.shape[0]

(original_count, filtered_count)

In [ ]:
# linear regression model with categorical variables
model3 = smf.ols('waso_min ~ C(day_after_flight_category)', data=df_no_outliers).fit()

print(model3.summary())

In [ ]:
# Ricostruiamo i modelli misti lineari con il dataset senza outlier
# Modello 1: Solo predittori principali
model_1_no_outliers = smf.mixedlm(
    "waso_min ~ C(location) + C(DST_0) + photoperiod",
    data=df_no_outliers,
    groups=df_no_outliers["day_after_flight"]
).fit()

# Modello 2: Aggiungiamo phase_sleepoffset_sunrise
model_2_no_outliers = smf.mixedlm(
    "waso_min ~ C(location) + C(DST_0) + photoperiod + C(location)*photoperiod",
    data=df_no_outliers,
    groups=df_no_outliers["day_after_flight"]
).fit()

# Modello 3: Aggiungiamo week_free_days
model_3_no_outliers = smf.mixedlm(
    "waso_min ~ C(location) + C(DST_0) + photoperiod + C(weekday_type)",
    data=df_no_outliers,
    groups=df_no_outliers["day_after_flight"]
).fit()

# Confrontiamo i modelli in termini di AIC e Log-Likelihood
model_comparison_no_outliers = pd.DataFrame({
    "Model": ["Base Model", "With interaction", "Plus weekdays"],
    "Log-Likelihood": [model_1_no_outliers.llf, model_2_no_outliers.llf, model_3_no_outliers.llf]
})

In [ ]:
model_comparison_no_outliers

In [ ]:
print(model_1_no_outliers.summary())

print(model_2_no_outliers.summary())

print(model_3_no_outliers.summary())

In [ ]:
coef_df4 = pd.DataFrame({'coef': model_3_no_outliers.params.values, 'p-value': model_3_no_outliers.pvalues.values, '0.025': model_3_no_outliers.conf_int()[0], '0.975': model_3_no_outliers.conf_int()[1]})

In [ ]:
# drop info not needed
coef_df4 = coef_df4.drop(index='Intercept')   
coef_df4 = coef_df4.drop('Group Var', axis=0)

In [ ]:
# initialize the matplotlib figure
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

# add a vertical line at x=0
plt.axvline(x=0, color='black', linewidth=1)

# create a strip plot for the coefficients
ax = sns.stripplot(x='coef', y=coef_df4.index, data=coef_df4, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)

# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_df4.shape[0]):
    plt.plot([coef_df4['0.025'].iloc[i], coef_df4['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_df4['0.025'].iloc[i], coef_df4['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_df4['0.975'].iloc[i], coef_df4['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add significance markers
for i in range(coef_df4.shape[0]):
    if coef_df4['p-value'].iloc[i] < 0.001:
        plt.text(coef_df4['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_df4['p-value'].iloc[i] < 0.01:
        plt.text(coef_df4['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_df4['p-value'].iloc[i] < 0.05:
        plt.text(coef_df4['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9)

# add a legend for significance markers
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('MixedLM n2 (outcome=phase "wake up time - sunrise")', fontsize=15, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.5)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


ax.set_yticklabels(['Location[T.UK]', 'Days after flight', 'Photoperiod', 'Location[T.UK]*Photoperiod', 'Time shift[T.ST]'])

plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='waso_min', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'])
plt.title('Waso by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('WASO (min)', fontsize=12, fontweight='bold')
plt.xlabel('')
plt.legend()
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.legend(title='Location', labels=['ITA', 'UK'], loc='upper right', handles=[plt.Line2D([0], [0], color='darkblue', lw=2), plt.Line2D([0], [0], color='darkorange')])
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST 
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

# mean of the phase for the two locations
#plt.axhline(y=df[df['location'] == 'ITA']['phase'].mean(), color='darkblue', linestyle='--', label='Mean ITA')
#plt.text(plt.xlim()[1], df[df['location'] == 'ITA']['phase'].mean(), 'Mean ITA', ha='right', va='bottom')
#plt.axhline(y=df[df['location'] == 'UK']['phase'].mean(), color='darkorange', linestyle='--', label='Mean UK')
#plt.text(plt.xlim()[1], df[df['location'] == 'UK']['phase'].mean(), 'Mean UK', ha='right', va='bottom')

plt.show()

_Phase (wake up time - sunrise)_

In [ ]:
# location coded as 0=ITA, 1=UK
#phase_bydays_after_flight['location'] = phase_bydays_after_flight['location'].map({'ITA': 0, 'UK': 1})

In [ ]:
# converting date to numerical values (days since the start of the observation period)
#data_phase['date_numeric'] = (pd.to_datetime(data_phase['date']) - pd.to_datetime(data_phase['date']).min()).dt.days

In [ ]:
# test the skewness of the phase data
skewness = skew(df['phase'])

print(f"Index of skewness: {skewness}")

In [ ]:
#plot the distribution
plt.figure(figsize=(6, 4))
sns.histplot(df['phase'], kde=True)
plt.title('Distribution of the scaled phase')
plt.xlabel('Scaled phase')
plt.ylabel('Frequency')
plt.show()

In [ ]:
phase_data = df['phase'].values.reshape(-1, 1)

# apply the Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
phase_transformed = pt.fit_transform(phase_data)

In [ ]:
# add the transformed phase to the dataframe 
df['phase_transformed'] = phase_transformed

In [ ]:
# plot the distribution of the transformed phase and the original phase
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['phase'], kde=True)
plt.title('Distribution of the phase')
plt.xlabel('Phase')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.histplot(phase_transformed, kde=True)
plt.title('Distribution of the transformed phase')
plt.xlabel('Transformed phase')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Shapiro-Wilk test for the transformed phase
shapiro_test4 = stats.shapiro(df['phase_transformed'])
shapiro_test4

In [ ]:
# Modello senza trasformazione
model_4a = smf.mixedlm('phase ~ C(location) + photoperiod + C(DST_0)',df, groups=df['day_after_flight']).fit()

model_4_summary = model_4a.summary()
model_4_summary

In [ ]:
# Modello senza trasformazione
model_4b = smf.mixedlm('phase ~ C(location) + photoperiod + C(DST_0) + C(location)*photoperiod',df, groups=df['day_after_flight']).fit()

model_4b_summary = model_4b.summary()
model_4b_summary

In [ ]:
coef_df4 = pd.DataFrame({'coef': model_4b.params.values, 'p-value': model_4b.pvalues.values, '0.025': model_4b.conf_int()[0], '0.975': model_4b.conf_int()[1]})

In [ ]:
# drop info not needed
coef_df4 = coef_df4.drop(index='Intercept')   
coef_df4 = coef_df4.drop('Group Var', axis=0)

In [ ]:
# initialize the matplotlib figure
plt.figure(figsize=(7, 5))
#sns.set_theme(style="whitegrid", rc={"axes.grid": False})  # set the style of the plot and remove the grid
#sns.set_palette("Paired")  # set the color palette of the plot
plt.axvline(x=0, color='#ae492a', linewidth=1, linestyle='--')  # add a vertical line at 0

# add a vertical line at x=0
plt.axvline(x=0, color='black', linewidth=1)

# create a strip plot for the coefficients
ax = sns.stripplot(x='coef', y=coef_df4.index, data=coef_df4, size=6, marker='D', linewidth=0.5, color='#265a69', edgecolor='#265a69', alpha=0.8)

# add the 0.025 and 0.975 confidence intervals as T-shaped lines
for i in range(coef_df4.shape[0]):
    plt.plot([coef_df4['0.025'].iloc[i], coef_df4['0.975'].iloc[i]], [i, i], color='dimgray', linewidth=1.5)
    plt.plot([coef_df4['0.025'].iloc[i], coef_df4['0.025'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)
    plt.plot([coef_df4['0.975'].iloc[i], coef_df4['0.975'].iloc[i]], [i - 0.01, i + 0.01], color='dimgray', linewidth=1.5)

# add significance markers
for i in range(coef_df4.shape[0]):
    if coef_df4['p-value'].iloc[i] < 0.001:
        plt.text(coef_df4['coef'].iloc[i], i, '***', ha='center', va='bottom', fontsize=9)
    elif coef_df4['p-value'].iloc[i] < 0.01:
        plt.text(coef_df4['coef'].iloc[i], i, '**', ha='center', va='bottom', fontsize=9)
    elif coef_df4['p-value'].iloc[i] < 0.05:
        plt.text(coef_df4['coef'].iloc[i], i, '*', ha='center', va='bottom', fontsize=9)

# add a legend for significance markers
plt.text(1.196, -0.1, 'p-value < 0.001 : ***', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.190, -0.05, 'p-value < 0.01 : ** ', ha='center', va='center', fontsize=10, transform=ax.transAxes)
plt.text(1.189, -0.00, 'p-value < 0.05 : *  ', ha='center', va='center', fontsize=10, transform=ax.transAxes)

ax.set_title('MixedLM (outcome=phase "wake up time - sunrise")', fontsize=15, fontweight='bold', pad=20)
plt.xlim(-2.75, 2.5)
plt.gca().xaxis.set_major_locator(MultipleLocator(0.5))
ax.spines['left'].set_color('dimgray')
ax.spines['bottom'].set_color('dimgray')
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
ax.set_xlabel('Estimates', fontsize=12)
ax.set_ylabel('Independent variables', fontsize=12)  # add y-axis title
# remove spines
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


ax.set_yticklabels(['Location[T.UK]', 'Time shift[T.ST]', 'Photoperiod', 'Location[T.UK]*Photoperiod'])

plt.show()

In [ ]:
# Shapiro-Wilk test for the residuals   
shapiro_residuals4 = stats.shapiro(model_4b.resid)

print('Shapiro test for the residuals:', shapiro_residuals4)

In [ ]:
residuals4 = model_4b.resid 
predicted4 = model_4b.fittedvalues

In [ ]:
# Breusch-Pagan test for homoscedasticity
bp_test4 = het_breuschpagan(residuals4, model_4b.model.exog)

# results of the Breusch-Pagan test
bp_stat4, bp_pval4, _, _ = bp_test4
print(f'Breusch-Pagan statistic: {bp_stat4}, p-value: {bp_pval4}')
if bp_pval4 > 0.05:
    print('The residuals are homoscedastic (fail to reject H0).')
else:
    print('The residuals are heteroscedastic (reject H0).')

In [ ]:
# Durbin-Watson test for autocorrelation
durbin_watson_test4 = durbin_watson(residuals4)

print('Durbin-Watson test:', durbin_watson_test4)

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='phase', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'])
plt.title('Phase by location over time', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Phase (wake up time-sunrise)')
plt.xlabel('')
plt.legend()
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min(), df['date'].max())
plt.gca().xaxis.set_major_locator(MultipleLocator(28)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST 
plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='phase', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'])
plt.title('Phase by location over time (2023)', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Phase (wake up time-sunrise)')
plt.xlabel('')
plt.legend()
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min() + pd.DateOffset(months=3), df['date'].min() + pd.DateOffset(months=15))
plt.gca().xaxis.set_major_locator(MultipleLocator(8)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST 
#plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
#plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
sns.scatterplot(x='date', y='phase', hue='location', style='weekday_type', data=df, palette=['darkblue', 'darkorange'])
plt.title('Phase by location over time (2024)', fontsize=16, fontweight='bold', loc='center', pad=20)
plt.ylabel('Phase (wake up time-sunrise)')
plt.xlabel('')
plt.legend()
plt.grid(True)
plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.xlim(df['date'].min() + pd.DateOffset(months=15), df['date'].min() + pd.DateOffset(months=27))
plt.gca().xaxis.set_major_locator(MultipleLocator(8)) #get the current axis = gca

# vertical bar to indicate the start of the DST and the start of the ST 
#plt.axvline(x=pd.to_datetime('2022-10-30'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2022-10-30'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
#plt.axvline(x=pd.to_datetime('2023-10-29'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2023-10-29'), plt.ylim()[0], 'start ST', ha='right', va='bottom')
#plt.axvline(x=pd.to_datetime('2023-03-26'), color='black', linestyle='--')
#plt.text(pd.to_datetime('2023-03-26'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-03-31'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-03-31'), plt.ylim()[0], 'start DST', ha='right', va='bottom')
plt.axvline(x=pd.to_datetime('2024-10-27'), color='black', linestyle='--')
plt.text(pd.to_datetime('2024-10-27'), plt.ylim()[0], 'start ST', ha='right', va='bottom')

# define seasons
seasons = [
    ('2022-09-22', '2022-12-21', 'brown'),  # Autumn
    ('2022-12-22', '2023-03-20', '#335f9e'),  # Winter
    ('2023-03-21', '2023-06-20', 'lightgreen'), # Spring
    ('2023-06-21', '2023-09-21', 'gold'),     # Summer
    ('2023-09-22', '2023-12-21', 'brown'),  # Autumn
    ('2023-12-22', '2024-03-20', '#335f9e'),  # Winter
    ('2024-03-21', '2024-06-20', 'lightgreen'), # Spring
    ('2024-06-21', '2024-09-21', 'gold'),     # Summer
    ('2024-09-22', '2024-12-21', 'brown'),  # Autumn
]

# apply the background color for each season
for start, end, color in seasons:
    plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=color, alpha=0.2)

plt.annotate('Summer', xy=(1.0555, 0.79), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='gold')
plt.annotate('Autumn', xy=(1.0535, 0.97), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='chocolate')
plt.annotate('Winter', xy=(1.05, 0.91), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#76d4f5')
plt.annotate('Spring', xy=(1.0499, 0.85), xycoords='axes fraction', ha='center', va='center', fontsize=12, backgroundcolor='#95de74')

plt.show()

##### _The re_formula parameter specifies the structure of the random effects in the mixed-effects model_
##### _re_formula-> specifies that the random effects are independent_

Common Usage

re_formula="~1":
This specifies that the random effects are independent and only include a random intercept for each group.
In other words, each group (defined by the groups parameter) has its own intercept, but the slopes are assumed to be the same across groups.

re_formula="~ variable":
This specifies that the random effects include both a random intercept and a random slope for the specified variable.
Each group has its own intercept and its own slope for the specified variable.

re_formula="~ variable1 + variable2":
This specifies that the random effects include a random intercept and random slopes for variable1 and variable2.
Each group has its own intercept and its own slopes for variable1 and variable2.